# LeetCode 446 — Arithmetic Slices II - Subsequence (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:

```

**Mục tiêu sau khi học xong:**

1. Hiểu đúng đề: phân biệt **subsequence (dãy con, được bỏ phần tử, giữ thứ tự)** với **subarray (mảng con liên tiếp)**, và hiểu vì sao đề đếm theo **vị trí được chọn**, không theo giá trị (nên `[7,7,7,7,7]` cho ra `16`).
2. Ôn lại bài dễ hơn **#413 Arithmetic Slices** (mảng con liên tiếp) từ căn bản, để thấy cái "khung" `dp[i] = số dãy kết thúc tại i`.
3. Viết **Cách 1 — brute force** bằng `itertools.combinations`: sinh mọi dãy con độ dài ≥ 3 rồi kiểm tra, và thấy vì sao nó là `O(2^n · n)`.
4. Học **đệ quy / backtracking** và viết **Cách 2 — DFS theo công sai chung**: chốt 2 phần tử đầu, rồi nối dài dần.
5. Học **memoization** và viết **Cách 3 — DFS + memo**, giảm xuống `O(n³)`.
6. Hiểu khái niệm then chốt **"dãy yếu" (weak arithmetic subsequence, độ dài ≥ 2)** và viết **Cách 4 — DP `dp[i][d]` bằng list các `defaultdict(int)`** chạy `O(n²)` — đây là lời giải chính.
7. Tự tay trace bảng DP cho `[2,4,6,8,10]` và `[7,7,7,7,7]`, nộp `class Solution`, test chéo với brute force, biết bẫy **tràn số ở C++/Java**.

## 1. Đề bài (theo LeetCode)

Cho một mảng số nguyên `nums`, hãy trả về **số lượng tất cả các dãy con cấp số cộng (arithmetic subsequences)** của `nums`.

Một dãy số được gọi là **cấp số cộng (arithmetic)** nếu nó có **ít nhất ba phần tử** và hiệu giữa hai phần tử liên tiếp bất kỳ đều **bằng nhau**.

- Ví dụ: `[1, 3, 5, 7, 9]`, `[7, 7, 7, 7]` và `[3, -1, -5, -9]` là các dãy cấp số cộng.
- Ví dụ: `[1, 1, 2, 5, 7]` **không** phải dãy cấp số cộng.

Một **dãy con (subsequence)** của mảng là dãy thu được bằng cách xóa đi một số phần tử (có thể không xóa phần tử nào) của mảng, **giữ nguyên thứ tự** các phần tử còn lại.

- Ví dụ: `[2, 5, 10]` là một dãy con của `[1, 2, 1, 2, 4, 1, 5, 10]`.

Các test case được sinh sao cho đáp án vừa với **số nguyên 32-bit**.

**Ví dụ 1**

```text
Input: nums = [2,4,6,8,10]
Output: 7
Giải thích: Tất cả các dãy con cấp số cộng là:
[2,4,6]
[4,6,8]
[6,8,10]
[2,4,6,8]
[4,6,8,10]
[2,4,6,8,10]
[2,6,10]
```

**Ví dụ 2**

```text
Input: nums = [7,7,7,7,7]
Output: 16
Giải thích: Mọi dãy con của mảng này đều là cấp số cộng.
```

**Constraints (ràng buộc):**

- `1 <= nums.length <= 1000`
- `-2^31 <= nums[i] <= 2^31 - 1`

Từ vựng: **công sai** (common difference, ký hiệu `d`) là hiệu cố định giữa hai số liên tiếp. Dãy `[2, 6, 10]` có công sai `d = 4`; dãy `[3, -1, -5]` có `d = -4`; dãy `[7, 7, 7]` có `d = 0`.

In [1]:
# Kiểm tra bằng mắt 7 dãy của Ví dụ 1: in từng dãy và các hiệu liên tiếp của nó
examples = [[2, 4, 6], [4, 6, 8], [6, 8, 10],
            [2, 4, 6, 8], [4, 6, 8, 10], [2, 4, 6, 8, 10], [2, 6, 10]]

for seq in examples:
    diffs = [seq[k + 1] - seq[k] for k in range(len(seq) - 1)]
    print(f"{str(seq):<18} hiệu liên tiếp = {diffs}")

print("Số dãy:", len(examples))

[2, 4, 6]          hiệu liên tiếp = [2, 2]
[4, 6, 8]          hiệu liên tiếp = [2, 2]
[6, 8, 10]         hiệu liên tiếp = [2, 2]
[2, 4, 6, 8]       hiệu liên tiếp = [2, 2, 2]
[4, 6, 8, 10]      hiệu liên tiếp = [2, 2, 2]
[2, 4, 6, 8, 10]   hiệu liên tiếp = [2, 2, 2, 2]
[2, 6, 10]         hiệu liên tiếp = [4, 4]
Số dãy: 7


### 1.1. Hiểu đúng 3 chi tiết dễ hiểu sai

**(a) Dãy con được "nhảy cóc", nhưng không được đảo thứ tự.**
`[2, 6, 10]` hợp lệ vì ta lấy các vị trí `0, 2, 4` của `[2,4,6,8,10]` theo đúng thứ tự tăng dần. Nhưng `[10, 8, 6]` thì **không** phải dãy con của `[2,4,6,8,10]` vì nó đi ngược thứ tự mảng.

Ngược lại, nếu mảng là `[1, 2, 3, 2, 1]` thì `[3, 2, 1]` (vị trí `2, 3, 4`) lại hợp lệ, công sai `d = -1`. Công sai âm hoàn toàn được phép.

**(b) Đếm theo VỊ TRÍ được chọn, không theo giá trị.**
Với `[7,7,7,7,7]`, ta chọn 3 vị trí `{0,1,2}` hay `{0,1,3}` đều ra dãy giá trị `[7,7,7]`, nhưng đó là **hai dãy con khác nhau** vì tập vị trí khác nhau. Vì vậy:

- số cách chọn 3 vị trí trong 5 = `C(5,3) = 10`
- số cách chọn 4 vị trí = `C(5,4) = 5`
- số cách chọn 5 vị trí = `C(5,5) = 1`

Tổng `10 + 5 + 1 = 16` — khớp đáp án Ví dụ 2.

**(c) Độ dài tối thiểu là 3.** Cặp hai phần tử `[2, 4]` **không** được đếm, dù "nhìn" có công sai. Chi tiết này sẽ là chìa khóa của lời giải chính (khái niệm "dãy yếu").

In [2]:
from math import comb   # comb(n, k) = số cách chọn k phần tử từ n phần tử

n = 5
for k in range(3, n + 1):
    print(f"C({n},{k}) = {comb(n, k)}")
print("Tổng =", sum(comb(n, k) for k in range(3, n + 1)))

C(5,3) = 10
C(5,4) = 5
C(5,5) = 1
Tổng = 16


## 2. Ôn bài dễ hơn: #413 Arithmetic Slices (mảng con LIÊN TIẾP)

Trước khi giải bài "dãy con", ta ôn bài anh em dễ hơn:

> **#413**: Đếm số **mảng con liên tiếp (subarray)** có độ dài ≥ 3 là cấp số cộng.

Ví dụ `nums = [1, 2, 3, 4]`:

- độ dài 3: `[1,2,3]`, `[2,3,4]`
- độ dài 4: `[1,2,3,4]`

→ đáp án `3`.

Ở #413 ta **không được nhảy cóc**: chỉ được cắt một đoạn liền nhau. Vì thế số đoạn chỉ có `O(n²)`, rất ít so với số dãy con `2^n`.

**Cách ngây thơ cho #413:** thử mọi đoạn `nums[l..r]` với `r - l + 1 >= 3`, kiểm tra từng đoạn có cùng hiệu hay không.

In [3]:
def arithmetic_slices_413_naive(nums):
    # Thử mọi đoạn liên tiếp nums[l..r] dài >= 3
    n = len(nums)
    count = 0
    for l in range(n):
        for r in range(l + 2, n):
            d = nums[l + 1] - nums[l]
            if all(nums[k + 1] - nums[k] == d for k in range(l, r)):
                count += 1
    return count

print(arithmetic_slices_413_naive([1, 2, 3, 4]))        # 3
print(arithmetic_slices_413_naive([1, 3, 5, 7, 9]))     # 6
print(arithmetic_slices_413_naive([1, 2, 4, 6, 8]))     # 3

3
6
3


### 2.1. DP cho #413: "đếm các đoạn KẾT THÚC tại i"

Ý tưởng DP (Dynamic Programming — quy hoạch động: lưu kết quả bài toán nhỏ để tính bài toán lớn) của #413:

> `dp[i]` = số mảng con cấp số cộng (dài ≥ 3) **kết thúc đúng tại vị trí `i`**.

Nếu `nums[i] - nums[i-1] == nums[i-1] - nums[i-2]` (ba số cuối cùng cùng hiệu) thì:

```python
dp[i] = dp[i-1] + 1
```

**Tại sao `+1`?** Mỗi đoạn tốt kết thúc tại `i-1` có thể "kéo dài" thêm `nums[i]` thành đoạn tốt kết thúc tại `i` (được `dp[i-1]` đoạn). Ngoài ra còn **một đoạn mới tinh** độ dài 3 là `[nums[i-2], nums[i-1], nums[i]]` → `+1`.

Ngược lại nếu ba số cuối không cùng hiệu thì `dp[i] = 0`.

Đáp án = `sum(dp)`.

Ví dụ `[1, 3, 5, 7, 9]`:

| i | nums[i] | hiệu cuối bằng hiệu trước? | dp[i] | ý nghĩa |
|---:|---:|:---:|---:|---|
| 0 | 1 | – | 0 | chưa đủ 3 số |
| 1 | 3 | – | 0 | chưa đủ 3 số |
| 2 | 5 | 2 == 2 ✔ | 1 | `[1,3,5]` |
| 3 | 7 | 2 == 2 ✔ | 2 | `[3,5,7]`, `[1,3,5,7]` |
| 4 | 9 | 2 == 2 ✔ | 3 | `[5,7,9]`, `[3,5,7,9]`, `[1,3,5,7,9]` |

Tổng = `0+0+1+2+3 = 6`.

In [4]:
def trace_arithmetic_slices_413(nums):
    n = len(nums)
    dp = [0] * n
    for i in range(2, n):
        same = nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]
        dp[i] = dp[i - 1] + 1 if same else 0
        print(f"i={i}, nums[i]={nums[i]}, cùng hiệu? {same!s:<5} -> dp[{i}] = {dp[i]}")
    print("dp =", dp, "| đáp án =", sum(dp))
    return sum(dp)

trace_arithmetic_slices_413([1, 3, 5, 7, 9])
print()
trace_arithmetic_slices_413([1, 2, 4, 6, 8])

i=2, nums[i]=5, cùng hiệu? True  -> dp[2] = 1
i=3, nums[i]=7, cùng hiệu? True  -> dp[3] = 2
i=4, nums[i]=9, cùng hiệu? True  -> dp[4] = 3
dp = [0, 0, 1, 2, 3] | đáp án = 6

i=2, nums[i]=4, cùng hiệu? False -> dp[2] = 0
i=3, nums[i]=6, cùng hiệu? True  -> dp[3] = 1
i=4, nums[i]=8, cùng hiệu? True  -> dp[4] = 2
dp = [0, 0, 0, 1, 2] | đáp án = 3


3

### 2.2. Từ #413 sang #446: cái gì thay đổi?

Hãy nhìn lại cái `+1` của #413. Đoạn mới `[nums[i-2], nums[i-1], nums[i]]` thực chất là: **lấy cặp hai phần tử `(i-2, i-1)` rồi nối thêm `nums[i]`**. Ở #413 ta không cần đếm cặp, vì với mỗi `i` **luôn có đúng một cặp** liền trước là `(i-2, i-1)` → nên chỉ cần `+1`.

Sang #446 (dãy con, được nhảy cóc), mọi thứ phức tạp lên ở 3 điểm:

1. **Phần tử đứng ngay trước `nums[i]` có thể là bất kỳ `nums[j]` với `j < i`**, không chỉ `nums[i-1]`.
2. **Mỗi lựa chọn `j` cho ra một công sai khác nhau** `d = nums[i] - nums[j]`. Kết thúc tại cùng một `i` có thể có dãy công sai 2, dãy công sai 4, ... Một con số `dp[i]` không đủ — ta cần **một bộ đếm cho từng công sai** → `dp[i][d]`.
3. **Số cặp có thể nối tiếp không còn là "đúng 1"**: kết thúc tại `j` với công sai `d` có thể có nhiều cặp `(k, j)` (ví dụ khi có giá trị trùng). Ta phải **đếm cả các cặp** — đây chính là lý do sinh ra khái niệm "dãy yếu" ở mục 6.

Nhưng trước khi tối ưu, ta đi đường của người mới: thử hết.

## 3. Kiến thức nền: `itertools.combinations` để sinh dãy con

Một dãy con được xác định hoàn toàn bởi **tập các vị trí được chọn, viết theo thứ tự tăng dần**. Python có sẵn công cụ sinh tất cả các tập vị trí như vậy:

```python
from itertools import combinations
combinations(iterable, k)   # sinh mọi bộ k phần tử, GIỮ thứ tự xuất hiện ban đầu
```

- `combinations(range(4), 3)` → `(0,1,2), (0,1,3), (0,2,3), (1,2,3)`: đúng 4 = `C(4,3)` bộ vị trí, mỗi bộ đã tăng dần.
- Vì sinh theo **vị trí**, nên hai số 7 ở hai vị trí khác nhau được coi là khác nhau — đúng tinh thần đếm của đề.

Từ một bộ vị trí `idx`, ta lấy dãy giá trị bằng list comprehension `[nums[t] for t in idx]`.

In [5]:
from itertools import combinations

nums = [2, 4, 6, 8]
for idx in combinations(range(len(nums)), 3):
    seq = [nums[t] for t in idx]
    print("vị trí", idx, "-> dãy", seq)

print("Số bộ 3 vị trí:", len(list(combinations(range(4), 3))))

vị trí (0, 1, 2) -> dãy [2, 4, 6]
vị trí (0, 1, 3) -> dãy [2, 4, 8]
vị trí (0, 2, 3) -> dãy [2, 6, 8]
vị trí (1, 2, 3) -> dãy [4, 6, 8]
Số bộ 3 vị trí: 4


### 3.1. Hàm kiểm tra một dãy có phải cấp số cộng

Một dãy `seq` (dài ≥ 3) là cấp số cộng khi **mọi** hiệu liên tiếp bằng hiệu đầu tiên `seq[1] - seq[0]`.

Hàm `all(...)` trả `True` nếu mọi phần tử trong dãy điều kiện là `True` (và dừng ngay khi gặp `False`).

In [6]:
def is_arithmetic(seq):
    if len(seq) < 3:
        return False                      # đề yêu cầu ít nhất 3 phần tử
    d = seq[1] - seq[0]
    return all(seq[k + 1] - seq[k] == d for k in range(len(seq) - 1))

print(is_arithmetic([1, 3, 5, 7, 9]))     # True
print(is_arithmetic([7, 7, 7, 7]))        # True  (d = 0)
print(is_arithmetic([3, -1, -5, -9]))     # True  (d = -4)
print(is_arithmetic([1, 1, 2, 5, 7]))     # False
print(is_arithmetic([2, 4]))              # False (chỉ 2 phần tử)

True
True
True
False
False


## 4. Cách 1 — Brute force: sinh mọi dãy con độ dài ≥ 3

Ý tưởng tự nhiên nhất:

```text
với mỗi độ dài L = 3, 4, ..., n:
    với mỗi bộ L vị trí tăng dần:
        nếu dãy giá trị là cấp số cộng -> đếm 1
```

Đây là cách "đúng theo định nghĩa", rất khó sai, nên ta sẽ dùng nó làm **trọng tài** để kiểm tra các cách nhanh hơn.

In [7]:
def arith_bruteforce(nums):
    n = len(nums)
    count = 0
    for length in range(3, n + 1):
        for idx in combinations(range(n), length):
            if is_arithmetic([nums[t] for t in idx]):
                count += 1
    return count

print(arith_bruteforce([2, 4, 6, 8, 10]))   # 7
print(arith_bruteforce([7, 7, 7, 7, 7]))    # 16
print(arith_bruteforce([1, 2, 3, 2, 1]))    # 2 : [1,2,3] và [3,2,1]
print(arith_bruteforce([1, 1, 2, 5, 7]))    # 0

7
16
2
0


### 4.1. Vì sao brute force bùng nổ?

Mỗi phần tử có 2 lựa chọn: **lấy** hoặc **không lấy** → có `2^n` dãy con. Bỏ đi những dãy dài 0, 1, 2 vẫn còn khoảng `2^n`. Mỗi dãy lại tốn tới `O(n)` để kiểm tra.

→ **Time `O(2^n · n)`**, Space `O(n)` cho một dãy đang xét.

Với `n = 1000` (giới hạn đề), `2^1000` là con số có hơn 300 chữ số — không máy nào chạy nổi. Ngay cả `n = 30` đã là hơn 1 tỉ dãy con. Cell dưới đo thử để cảm nhận tốc độ tăng: mỗi lần `n` tăng 2, thời gian xấp xỉ ×4.

In [8]:
import time, random

random.seed(0)
for n in [10, 12, 14, 16]:
    arr = [random.randint(-5, 5) for _ in range(n)]
    t0 = time.perf_counter()
    ans = arith_bruteforce(arr)
    t1 = time.perf_counter()
    print(f"n={n:>2} | số dãy con dài>=3 = {2**n - 1 - n - comb(n, 2):>6} | đáp án = {ans:>4} | {t1 - t0:.3f}s")

n=10 | số dãy con dài>=3 =    968 | đáp án =    6 | 0.001s
n=12 | số dãy con dài>=3 =   4017 | đáp án =    4 | 0.002s
n=14 | số dãy con dài>=3 =  16278 | đáp án =   21 | 0.009s
n=16 | số dãy con dài>=3 =  65399 | đáp án =   22 | 0.042s


**Nút thắt của Cách 1:** ta sinh cả những dãy **chắc chắn hỏng từ sớm**. Ví dụ với `[2, 4, 7, ...]`: ngay khi thấy `4-2 = 2` mà `7-4 = 3`, mọi dãy con bắt đầu bằng `2, 4, 7` đều hỏng — nhưng brute force vẫn sinh hết chúng rồi mới kiểm tra.

**Cải tiến tự nhiên:** đừng sinh bừa. Hãy **chốt công sai ngay từ 2 phần tử đầu**, rồi chỉ đi tiếp tới những phần tử **đúng** công sai đó. Đó là Cách 2.

## 5. Kiến thức nền: đệ quy (recursion) và backtracking

**Đệ quy** là khi một hàm **tự gọi lại chính nó** với bài toán nhỏ hơn. Mỗi hàm đệ quy cần:

1. **Trường hợp dừng (base case)**: khi nào không gọi tiếp.
2. **Bước đệ quy**: giải phần nhỏ hơn rồi ghép kết quả.

**Backtracking / DFS (Depth-First Search — tìm kiếm theo chiều sâu)**: đi sâu theo một lựa chọn cho tới cùng, rồi quay lui thử lựa chọn khác.

Ví dụ mini: liệt kê mọi dãy con **bắt đầu tại vị trí `start`** của `[2, 4, 6]` bằng cách: in dãy hiện tại, rồi thử nối thêm từng vị trí `k > last`.

In [9]:
def list_subsequences_from(nums, path, last, depth=0):
    # path: các giá trị đã chọn; last: vị trí cuối cùng đã chọn
    print("  " * depth + str(path))
    for k in range(last + 1, len(nums)):
        list_subsequences_from(nums, path + [nums[k]], k, depth + 1)   # đi sâu
        # khi lời gọi trên trả về, ta "quay lui" và thử k tiếp theo

list_subsequences_from([2, 4, 6], [2], 0)

[2]
  [2, 4]
    [2, 4, 6]
  [2, 6]


## 6. Cách 2 — DFS theo công sai chung

**Quan sát:** một cấp số cộng được **xác định hoàn toàn bởi 2 phần tử đầu tiên** của nó: phần tử đầu `nums[i]` và công sai `d = nums[j] - nums[i]`. Từ đó các phần tử tiếp theo **bắt buộc** phải bằng `nums[j] + d`, `nums[j] + 2d`, ...

Vậy ta làm như sau:

1. Duyệt mọi cặp `i < j` làm **2 phần tử đầu**, công sai `d = nums[j] - nums[i]`.
2. Gọi `extend(last=j, d)` = **số cách nối dài** dãy đang kết thúc tại vị trí `last` với công sai `d` (nối thêm ít nhất 1 phần tử).
3. Trong `extend(last, d)`: với mỗi `k > last` mà `nums[k] - nums[last] == d`:
   - nối `nums[k]` vào → được **1 dãy mới hợp lệ** (vì nó đã có ≥ 3 phần tử) → `+1`;
   - và từ `k` còn nối dài tiếp được `extend(k, d)` dãy nữa.

```text
extend(last, d) = Σ  [ 1 + extend(k, d) ]   với k > last và nums[k] - nums[last] == d
```

Base case ngầm: nếu không có `k` nào thỏa thì vòng lặp không chạy và trả `0`.

Ví dụ `[2,4,6,8,10]`, cặp đầu `(2, 4)` → `d = 2`: nối được `[2,4,6]`, `[2,4,6,8]`, `[2,4,6,8,10]` → `extend(1, 2) = 3`.

In [10]:
def arith_dfs(nums):
    n = len(nums)

    def extend(last, d):
        total = 0
        for k in range(last + 1, n):
            if nums[k] - nums[last] == d:
                total += 1 + extend(k, d)      # 1 dãy vừa tạo + các dãy dài hơn nữa
        return total

    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            ans += extend(j, nums[j] - nums[i])   # (i, j) là 2 phần tử đầu
    return ans

print(arith_dfs([2, 4, 6, 8, 10]))   # 7
print(arith_dfs([7, 7, 7, 7, 7]))    # 16
print(arith_dfs([1, 2, 3, 2, 1]))    # 2

7
16
2


### 6.1. Trace DFS trên `[2,4,6,8,10]`

Hàm `trace_extend` in ra cây gọi đệ quy (thụt lề = độ sâu) để thấy rõ từng lần "nối thêm 1 phần tử = thêm 1 dãy". Ta trace cho cặp đầu `(2,4)` và `(2,6)`, rồi in đóng góp của **mọi** cặp.

Kỳ vọng (tính tay):

| cặp đầu (giá trị) | d | các dãy được tạo | đóng góp |
|---|---:|---|---:|
| `(2,4)` | 2 | `[2,4,6]`, `[2,4,6,8]`, `[2,4,6,8,10]` | 3 |
| `(2,6)` | 4 | `[2,6,10]` | 1 |
| `(4,6)` | 2 | `[4,6,8]`, `[4,6,8,10]` | 2 |
| `(6,8)` | 2 | `[6,8,10]` | 1 |
| các cặp còn lại | – | không nối được | 0 |

Tổng = `3 + 1 + 2 + 1 = 7`.

In [11]:
def trace_extend(nums, path, last, d, depth=1):
    total = 0
    for k in range(last + 1, len(nums)):
        if nums[k] - nums[last] == d:
            new_path = path + [nums[k]]
            print("  " * depth + f"nối {nums[k]} -> dãy mới {new_path}")
            total += 1 + trace_extend(nums, new_path, k, d, depth + 1)
    return total

nums = [2, 4, 6, 8, 10]
for i, j in [(0, 1), (0, 2)]:
    d = nums[j] - nums[i]
    print(f"Cặp đầu ({nums[i]},{nums[j]}), d={d}:")
    got = trace_extend(nums, [nums[i], nums[j]], j, d)
    print(f"  => đóng góp {got}")

Cặp đầu (2,4), d=2:
  nối 6 -> dãy mới [2, 4, 6]
    nối 8 -> dãy mới [2, 4, 6, 8]
      nối 10 -> dãy mới [2, 4, 6, 8, 10]
  => đóng góp 3
Cặp đầu (2,6), d=4:
  nối 10 -> dãy mới [2, 6, 10]
  => đóng góp 1


Để in đóng góp của **mọi** cặp đầu mà không bị in cây gọi quá dài, ta dùng một bản `extend` "im lặng" (không print) và chỉ in những cặp có đóng góp > 0. Kết quả phải khớp bảng tính tay ở trên.

In [12]:
def extend_quiet(nums, last, d):
    total = 0
    for k in range(last + 1, len(nums)):
        if nums[k] - nums[last] == d:
            total += 1 + extend_quiet(nums, k, d)
    return total

nums = [2, 4, 6, 8, 10]
total = 0
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        d = nums[j] - nums[i]
        c = extend_quiet(nums, j, d)
        total += c
        if c > 0:
            print(f"cặp ({nums[i]},{nums[j]}) d={d}: đóng góp {c}")
print("Tổng =", total)

cặp (2,4) d=2: đóng góp 3
cặp (2,6) d=4: đóng góp 1
cặp (4,6) d=2: đóng góp 2
cặp (6,8) d=2: đóng góp 1
Tổng = 7


### 6.2. DFS vẫn chậm ở đâu?

DFS đã **không sinh dãy hỏng** nữa: nó chỉ đi tới những `k` đúng công sai. Với mảng "lộn xộn" thì rất nhanh.

Nhưng hãy xem trường hợp xấu nhất `[7, 7, ..., 7]`: **mọi** dãy con đều hợp lệ, nên đáp án đã là ~`2^n`, và DFS đếm từng dãy một (mỗi dãy là `+1`) → vẫn **hàm mũ**.

Quan trọng hơn: nhìn kỹ, **cùng một lời gọi `extend(last, d)` bị tính đi tính lại**. Với `[2,4,6,8,10]`: `extend(2, 2)` (đang ở số 6, công sai 2) được gọi từ cặp đầu `(2,4)` (qua `extend(1,2)`) **và** từ cặp đầu `(4,6)`. Kết quả hai lần như nhau vì nó chỉ phụ thuộc `(last, d)`, không phụ thuộc phần đầu dãy.

Cell dưới đếm số lần gọi `extend` trên `[7]*n`.

In [13]:
def count_dfs_calls(nums):
    n = len(nums)
    calls = 0

    def extend(last, d):
        nonlocal calls            # nonlocal: sửa biến calls của hàm bao bên ngoài
        calls += 1
        total = 0
        for k in range(last + 1, n):
            if nums[k] - nums[last] == d:
                total += 1 + extend(k, d)
        return total

    ans = sum(extend(j, nums[j] - nums[i]) for i in range(n) for j in range(i + 1, n))
    return ans, calls

for n in [8, 10, 12, 14]:
    ans, calls = count_dfs_calls([7] * n)
    print(f"n={n:>2} | đáp án = {ans:>6} | số lần gọi extend = {calls:>6}")

n= 8 | đáp án =    219 | số lần gọi extend =    247
n=10 | đáp án =    968 | số lần gọi extend =   1013
n=12 | đáp án =   4017 | số lần gọi extend =   4083
n=14 | đáp án =  16278 | số lần gọi extend =  16369


## 7. Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = lần đầu tính xong một bài toán con thì **lưu kết quả vào dict**; lần sau gặp lại cùng tham số thì **trả luôn**, không tính lại.

Ví dụ kinh điển: dãy Fibonacci `fib(n) = fib(n-1) + fib(n-2)`. Bản đệ quy thường gọi `fib(30)` hơn 1,6 triệu lần; bản có memo chỉ tính mỗi `n` đúng một lần.

```python
memo = {}
def fib(n):
    if n in memo:            # đã tính rồi -> trả luôn
        return memo[n]
    ...tính...
    memo[n] = kết_quả        # lưu lại
```

(Python còn có `functools.lru_cache` làm việc này tự động, nhưng tự viết dict giúp ta nhìn rõ cơ chế.)

In [14]:
calls_plain = 0
def fib_plain(n):
    global calls_plain
    calls_plain += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

memo, calls_memo = {}, 0
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    if n in memo:
        return memo[n]
    memo[n] = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

print("fib(25) thường :", fib_plain(25), "| số lần gọi:", calls_plain)
print("fib(25) có memo:", fib_memo(25), "| số lần gọi:", calls_memo)

fib(25) thường : 75025 | số lần gọi: 242785
fib(25) có memo: 75025 | số lần gọi: 49


## 8. Cách 3 — DFS + memo theo trạng thái `(last, d)`

Như đã thấy ở 6.2, `extend(last, d)` chỉ phụ thuộc **vị trí cuối `last`** và **công sai `d`**. Vậy ta lưu `memo[(last, d)]`.

- **Số trạng thái:** `last` có `n` giá trị; với mỗi `last`, những `d` thực sự xuất hiện là `nums[last] - nums[i]` với `i < last` → tối đa `n` giá trị. Vậy ≤ `n²` trạng thái.
- **Mỗi trạng thái** quét vòng `for k` mất `O(n)`.

→ **Time `O(n³)`**, Space `O(n²)` cho memo (+ `O(n)` độ sâu đệ quy).

Lưu ý dùng **tuple `(last, d)` làm key** của dict: tuple là kiểu bất biến (immutable) nên dùng làm key được, còn list thì không.

In [15]:
def arith_dfs_memo(nums, stats=None):
    n = len(nums)
    memo = {}

    def extend(last, d):
        key = (last, d)
        if key in memo:
            return memo[key]
        total = 0
        for k in range(last + 1, n):
            if nums[k] - nums[last] == d:
                total += 1 + extend(k, d)
        memo[key] = total
        return total

    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            ans += extend(j, nums[j] - nums[i])
    if stats is not None:
        stats["states"] = len(memo)
    return ans

print(arith_dfs_memo([2, 4, 6, 8, 10]))   # 7
print(arith_dfs_memo([7, 7, 7, 7, 7]))    # 16

for n in [8, 10, 12, 14]:
    st = {}
    ans = arith_dfs_memo([7] * n, st)
    print(f"n={n:>2} | đáp án = {ans:>6} | số trạng thái memo = {st['states']}")

7
16
n= 8 | đáp án =    219 | số trạng thái memo = 7
n=10 | đáp án =    968 | số trạng thái memo = 9
n=12 | đáp án =   4017 | số trạng thái memo = 11
n=14 | đáp án =  16278 | số trạng thái memo = 13


### 8.1. Vẫn chưa đủ cho `n = 1000`

- `O(n³)` với `n = 1000` là khoảng `10^9` phép kiểm tra `nums[k] - nums[last] == d` → quá chậm với Python (và sát giới hạn cả với C++).
- Đệ quy sâu tới `n` tầng; Python mặc định giới hạn khoảng 1000 tầng (`sys.getrecursionlimit()`), dễ dính `RecursionError` với chuỗi dài.

**Chỗ lãng phí còn lại:** trong `extend(last, d)` ta quét **toàn bộ** `k > last` chỉ để tìm vài `k` đúng công sai. Nếu ta **đổi chiều nhìn** — thay vì "từ `last` nhìn **về sau** xem nối được ai", ta đứng ở `i` **nhìn về trước**, giống #413 — thì mỗi cặp `(j, i)` chỉ cần xét **đúng một lần**, và ta có `O(n²)`.

In [16]:
import sys
print("Giới hạn độ sâu đệ quy mặc định:", sys.getrecursionlimit())

Giới hạn độ sâu đệ quy mặc định: 1000


## 9. Bước đột phá: nhìn về trước và khái niệm "dãy yếu"

### 9.1. Thử bắt chước #413 một cách ngây thơ

Bắt chước #413, ta muốn định nghĩa:

> `real[i][d]` = số dãy cấp số cộng **thật** (dài ≥ 3) kết thúc tại `i`, công sai `d`.

Xét một dãy thật kết thúc tại `i` và phần tử kề trước nó là `j` (`d = nums[i] - nums[j]`). Bỏ `nums[i]` đi, phần còn lại kết thúc tại `j` có thể là:

- **(a)** một dãy thật (dài ≥ 3) kết thúc tại `j` với công sai `d` → có `real[j][d]` cái;
- **(b)** **chỉ là một cặp** `(k, j)` với `nums[j] - nums[k] = d` → dãy mới độ dài 3 là `(k, j, i)`.

Trường hợp (b) cho thấy: **chỉ đếm dãy thật là không đủ**, ta còn phải biết **số cặp** kết thúc tại `j` với công sai `d`. Ở #413 số cặp này luôn là 1 nên được giấu vào `+1`; ở #446 nó có thể là 0, 1, 2, ... (ví dụ `[2, 2, 3]`: kết thúc tại số 3 với `d = 1` có **hai** cặp `(2,3)` vì có hai số 2).

### 9.2. Gộp lại: "dãy yếu"

Thay vì giữ hai bộ đếm (dãy thật + cặp), ta gộp chúng làm một:

> **Dãy yếu (weak arithmetic subsequence)** = dãy con có **độ dài ≥ 2** mà các hiệu liên tiếp bằng nhau.
>
> Mọi cặp 2 phần tử đều là dãy yếu; mọi dãy thật cũng là dãy yếu.
>
> `dp[i][d]` = số **dãy yếu** kết thúc tại vị trí `i` với công sai `d` = `real[i][d]` + (số cặp kết thúc tại `i` với công sai `d`).

Bây giờ phát biểu (a)+(b) gọn lại thành một câu:

> Lấy **bất kỳ dãy yếu** nào kết thúc tại `j` với công sai `d`, nối thêm `nums[i]` (`d = nums[i] - nums[j]`) → luôn được một dãy **dài ≥ 3** = **dãy thật**.

Vì vậy số dãy thật có **hai phần tử cuối là `(j, i)`** đúng bằng `dp[j][d]`. Đây là con số ta cộng vào đáp án. Dãy yếu chỉ là "nguyên liệu trung gian" để không bỏ sót các dãy độ dài 3 — bản thân nó (khi dài 2) **không bao giờ** được cộng vào đáp án.

In [17]:
# Liệt kê bằng brute force mọi DÃY YẾU (dài >= 2) của [2,4,6,8], nhóm theo (vị trí cuối, công sai)
from collections import defaultdict

def weak_sequences_by_end(nums):
    n = len(nums)
    groups = defaultdict(list)
    for length in range(2, n + 1):
        for idx in combinations(range(n), length):
            seq = [nums[t] for t in idx]
            d = seq[1] - seq[0]
            if all(seq[k + 1] - seq[k] == d for k in range(len(seq) - 1)):
                groups[(idx[-1], d)].append(seq)
    return groups

groups = weak_sequences_by_end([2, 4, 6, 8])
for (end, d), seqs in sorted(groups.items()):
    print(f"kết thúc tại vị trí {end}, d={d}: {len(seqs)} dãy yếu -> {seqs}")

kết thúc tại vị trí 1, d=2: 1 dãy yếu -> [[2, 4]]
kết thúc tại vị trí 2, d=2: 2 dãy yếu -> [[4, 6], [2, 4, 6]]
kết thúc tại vị trí 2, d=4: 1 dãy yếu -> [[2, 6]]
kết thúc tại vị trí 3, d=2: 3 dãy yếu -> [[6, 8], [4, 6, 8], [2, 4, 6, 8]]
kết thúc tại vị trí 3, d=4: 1 dãy yếu -> [[4, 8]]
kết thúc tại vị trí 3, d=6: 1 dãy yếu -> [[2, 8]]


Đọc kết quả trên: kết thúc tại vị trí 2 (số `6`) với `d=2` có **2 dãy yếu** `[4,6]` và `[2,4,6]`. Nối số `8` (vị trí 3) vào **cả hai** → `[4,6,8]` và `[2,4,6,8]`: đúng **2 dãy thật** có hai phần tử cuối là `(6, 8)`.

Cell dưới kiểm chứng câu khẳng định này cho **mọi** cặp `(j, i)` trên một mảng có cả giá trị trùng: số dãy thật có hai phần tử cuối `(j, i)` (đếm bằng brute force) == số dãy yếu kết thúc tại `j` với công sai `nums[i]-nums[j]`.

In [18]:
def check_weak_claim(nums):
    n = len(nums)
    weak = weak_sequences_by_end(nums)
    ok = True
    for i in range(n):
        for j in range(i):
            d = nums[i] - nums[j]
            # số dãy thật (dài >= 3) mà 2 vị trí cuối là (j, i)
            real_ending_ji = 0
            for length in range(3, n + 1):
                for idx in combinations(range(n), length):
                    if idx[-2:] == (j, i) and is_arithmetic([nums[t] for t in idx]):
                        real_ending_ji += 1
            weak_at_j = len(weak.get((j, d), []))
            ok &= (real_ending_ji == weak_at_j)
    return ok

print(check_weak_claim([2, 4, 6, 8]))
print(check_weak_claim([2, 2, 3, 4, 4, 5]))
print(check_weak_claim([7, 7, 7, 7, 7]))

True
True
True


### 9.3. Công thức chuyển trạng thái

Duyệt `i` từ trái sang phải; với mỗi `j < i`, đặt `d = nums[i] - nums[j]`:

```python
ans      += dp[j][d]          # (1) mọi dãy yếu tại j nối thêm nums[i] -> dãy THẬT -> đếm
dp[i][d] += dp[j][d] + 1      # (2) các dãy yếu mới kết thúc tại i:
                              #     - dp[j][d] dãy vừa nối ở trên (giờ kết thúc tại i)
                              #     - +1 là cặp mới (j, i)
```

So với #413: dòng (2) chính là `dp[i] = dp[i-1] + 1` nhưng **tách theo công sai** và **cộng dồn qua mọi `j`** (dùng `+=` chứ không phải `=`, vì nhiều `j` khác nhau có thể cho cùng `d` — khi có giá trị trùng).

Điểm khác cốt lõi: ở #413 `dp` đếm thẳng dãy thật nên đáp án là `sum(dp)`; ở #446 `dp` đếm **dãy yếu**, nên đáp án **không phải** tổng `dp`, mà là tổng của các phần **được nối thêm** ở dòng (1). Đây là cách loại bỏ tự nhiên các dãy độ dài 2.

## 10. Kiến thức nền: `dict`, `defaultdict(int)` và list các dict

Công sai `d` có thể là bất kỳ số nào trong khoảng `[-(2^32 - 1), 2^32 - 1]` (rất rộng, có cả số âm) nên **không thể** dùng mảng 2 chiều `dp[i][d]` đánh chỉ số bằng `d`. Ta dùng **dict**: mỗi vị trí `i` có một dict riêng `dp[i]` ánh xạ `công sai -> số dãy yếu`. Dict chỉ chứa những `d` thực sự xuất hiện.

`collections.defaultdict(int)` là dict mà khi **ghi** vào một key chưa có, nó tự coi giá trị cũ là `int() = 0`. Nhờ vậy viết `dp[i][d] += 1` không lo `KeyError`.

```python
from collections import defaultdict
c = defaultdict(int)
c[5] += 1        # không lỗi: c[5] mặc định 0 rồi +1
```

**Mẹo đọc an toàn:** khi chỉ muốn **đọc**, dùng `dp[j].get(d, 0)`. Nếu viết `dp[j][d]` với defaultdict, Python sẽ **chèn thêm** key `d` với giá trị 0 vào `dp[j]` (tác dụng phụ), làm dict phình to vô ích.

In [19]:
from collections import defaultdict

plain = {}
# plain[5] += 1         # sẽ lỗi KeyError vì key 5 chưa có
plain[5] = plain.get(5, 0) + 1
print("dict thường     :", plain)

c = defaultdict(int)
c[5] += 1
c[-3] += 2
print("defaultdict     :", dict(c))

# Đọc bằng [] trên defaultdict sẽ CHÈN key mới:
_ = c[100]
print("sau khi đọc c[100]:", dict(c), "<- có thêm key 100 với giá trị 0")
print("đọc bằng get      :", c.get(999, 0), "| dict vẫn:", dict(c))

dict thường     : {5: 1}
defaultdict     : {5: 1, -3: 2}
sau khi đọc c[100]: {5: 1, -3: 2, 100: 0} <- có thêm key 100 với giá trị 0
đọc bằng get      : 0 | dict vẫn: {5: 1, -3: 2, 100: 0}


### 10.1. Tạo list gồm `n` dict độc lập — và cái bẫy `[defaultdict(int)] * n`

Ta cần `dp = [dict cho vị trí 0, dict cho vị trí 1, ...]`. Cách đúng:

```python
dp = [defaultdict(int) for _ in range(n)]     # n đối tượng KHÁC NHAU
```

Cách **sai** rất hay gặp:

```python
dp = [defaultdict(int)] * n                   # n tham chiếu tới CÙNG MỘT dict!
```

Phép `* n` chỉ nhân bản **tham chiếu (reference)**, nên sửa `dp[1]` cũng là sửa `dp[0]`, `dp[2]`, ... Cell dưới cho thấy rõ.

In [20]:
n = 3
dp_wrong = [defaultdict(int)] * n
dp_right = [defaultdict(int) for _ in range(n)]

dp_wrong[1][2] += 1
dp_right[1][2] += 1

print("sai :", [dict(x) for x in dp_wrong], "| cùng 1 object?", dp_wrong[0] is dp_wrong[2])
print("đúng:", [dict(x) for x in dp_right], "| cùng 1 object?", dp_right[0] is dp_right[2])

sai : [{2: 1}, {2: 1}, {2: 1}] | cùng 1 object? True
đúng: [{}, {2: 1}, {}] | cùng 1 object? False


## 11. Cách 4 — DP `dp[i][d]` với dãy yếu, `O(n²)` (lời giải chính)

Ghép mọi mảnh lại:

```text
dp = [defaultdict(int) for _ in range(n)]
ans = 0
for i in 0..n-1:
    for j in 0..i-1:
        d = nums[i] - nums[j]
        cnt = dp[j].get(d, 0)      # số dãy yếu kết thúc tại j, công sai d
        ans += cnt                 # nối nums[i] -> cnt dãy thật mới
        dp[i][d] += cnt + 1        # dãy yếu kết thúc tại i: cnt dãy nối + 1 cặp (j, i)
return ans
```

Vì sao thứ tự duyệt `i` tăng dần là đủ? Khi xử lý `i`, ta chỉ đọc `dp[j]` với `j < i` — những dict này **đã hoàn chỉnh** (mọi `j' < j` đã được xét khi ta xử lý `j`). Còn `dp[i]` chỉ được ghi, không bị đọc trong vòng của chính nó.

In [21]:
def arith_dp(nums):
    n = len(nums)
    dp = [defaultdict(int) for _ in range(n)]   # dp[i][d] = số dãy yếu kết thúc tại i, công sai d
    ans = 0
    for i in range(n):
        for j in range(i):
            d = nums[i] - nums[j]
            cnt = dp[j].get(d, 0)
            ans += cnt                 # chỉ phần "nối dài" mới là dãy dài >= 3
            dp[i][d] += cnt + 1        # +1: cặp (j, i)
    return ans

print(arith_dp([2, 4, 6, 8, 10]))   # 7
print(arith_dp([7, 7, 7, 7, 7]))    # 16
print(arith_dp([1, 2, 3, 2, 1]))    # 2
print(arith_dp([1, 1, 2, 5, 7]))    # 0
print(arith_dp([2, 2, 3, 4]))       # 2 : [2,3,4] với số 2 ở vị trí 0 hoặc 1

7
16
2
0
2


## 12. Trace từng bước trên `[2, 4, 6, 8, 10]`

Mỗi dòng là một cặp `(j, i)`. Cột "dp[j][d] trước" là số dãy yếu tại `j` được nối thêm `nums[i]`.

| i (nums[i]) | j (nums[j]) | d | dp[j][d] trước | ans sau | dp[i][d] sau | ghi chú |
|---|---|---:|---:|---:|---:|---|
| 1 (4) | 0 (2) | 2 | 0 | 0 | 1 | chỉ có cặp `[2,4]` |
| 2 (6) | 0 (2) | 4 | 0 | 0 | 1 | cặp `[2,6]` |
| 2 (6) | 1 (4) | 2 | 1 | **1** | 2 | nối `[2,4]` → `[2,4,6]` ✔ |
| 3 (8) | 0 (2) | 6 | 0 | 1 | 1 | cặp `[2,8]` |
| 3 (8) | 1 (4) | 4 | 0 | 1 | 1 | cặp `[4,8]` |
| 3 (8) | 2 (6) | 2 | 2 | **3** | 3 | nối `[4,6]`, `[2,4,6]` → `[4,6,8]`, `[2,4,6,8]` |
| 4 (10) | 0 (2) | 8 | 0 | 3 | 1 | cặp `[2,10]` |
| 4 (10) | 1 (4) | 6 | 0 | 3 | 1 | cặp `[4,10]` |
| 4 (10) | 2 (6) | 4 | 1 | **4** | 2 | nối `[2,6]` → `[2,6,10]` |
| 4 (10) | 3 (8) | 2 | 3 | **7** | 4 | nối `[6,8]`, `[4,6,8]`, `[2,4,6,8]` → 3 dãy |

Đáp án cuối `ans = 7` ✔. Hàm `trace_arith_dp` dưới in đúng bảng này.

In [22]:
def trace_arith_dp(nums, show_zero=True):
    n = len(nums)
    dp = [defaultdict(int) for _ in range(n)]
    ans = 0
    print(f"{'i(val)':>7} {'j(val)':>7} {'d':>4} {'dp[j][d]':>9} {'ans':>5} {'dp[i][d]':>9}")
    for i in range(n):
        for j in range(i):
            d = nums[i] - nums[j]
            cnt = dp[j].get(d, 0)
            ans += cnt
            dp[i][d] += cnt + 1
            if show_zero or cnt:
                print(f"{i:>3}({nums[i]:>2}) {j:>3}({nums[j]:>2}) {d:>4} {cnt:>9} {ans:>5} {dp[i][d]:>9}")
    print("Đáp án:", ans)
    return dp, ans

dp_table, _ = trace_arith_dp([2, 4, 6, 8, 10])

 i(val)  j(val)    d  dp[j][d]   ans  dp[i][d]
  1( 4)   0( 2)    2         0     0         1
  2( 6)   0( 2)    4         0     0         1
  2( 6)   1( 4)    2         1     1         2
  3( 8)   0( 2)    6         0     1         1
  3( 8)   1( 4)    4         0     1         1
  3( 8)   2( 6)    2         2     3         3
  4(10)   0( 2)    8         0     3         1
  4(10)   1( 4)    6         0     3         1
  4(10)   2( 6)    4         1     4         2
  4(10)   3( 8)    2         3     7         4
Đáp án: 7


### 12.1. Bảng `dp` cuối cùng và ý nghĩa

| i | nums[i] | dp[i] |
|---:|---:|---|
| 0 | 2 | `{}` |
| 1 | 4 | `{2: 1}` |
| 2 | 6 | `{4: 1, 2: 2}` |
| 3 | 8 | `{6: 1, 4: 1, 2: 3}` |
| 4 | 10 | `{8: 1, 6: 1, 4: 2, 2: 4}` |

Đọc ví dụ `dp[4][2] = 4`: có 4 dãy yếu kết thúc tại số 10 với công sai 2: `[8,10]`, `[6,8,10]`, `[4,6,8,10]`, `[2,4,6,8,10]` — đúng **1 cặp + 3 dãy thật**. Còn `dp[4][4] = 2`: `[6,10]` và `[2,6,10]`.

Để ý: `dp[4]` (phần tử cuối) **không bao giờ được cộng vào đáp án** — vì không còn ai đứng sau để nối. Những dãy thật trong `dp[4]` đã được đếm vào `ans` ngay lúc chúng được tạo ra.

In [23]:
for i, row in enumerate(dp_table):
    print(f"dp[{i}] (nums[{i}]={[2, 4, 6, 8, 10][i]:>2}) = {dict(row)}")

dp[0] (nums[0]= 2) = {}
dp[1] (nums[1]= 4) = {2: 1}
dp[2] (nums[2]= 6) = {4: 1, 2: 2}
dp[3] (nums[3]= 8) = {6: 1, 4: 1, 2: 3}
dp[4] (nums[4]=10) = {8: 1, 6: 1, 4: 2, 2: 4}


### 12.2. Trace trường hợp giá trị trùng `[7, 7, 7, 7, 7]`

Mọi công sai đều là `d = 0`, và mỗi `i` nhận đóng góp từ **mọi** `j < i` → đây là lúc dấu `+=` (cộng dồn qua nhiều `j`) thể hiện vai trò.

| i | các j | dp[j][0] lần lượt | ans tăng thêm | ans sau | dp[i][0] sau |
|---:|---|---|---:|---:|---:|
| 1 | 0 | 0 | 0 | 0 | 1 |
| 2 | 0, 1 | 0, 1 | 1 | 1 | 1 + 2 = 3 |
| 3 | 0, 1, 2 | 0, 1, 3 | 4 | 5 | 1 + 2 + 4 = 7 |
| 4 | 0, 1, 2, 3 | 0, 1, 3, 7 | 11 | 16 | 1 + 2 + 4 + 8 = 15 |

Nhận xét: `dp[i][0] = 2^i - 1` = số cách chọn **một tập con khác rỗng** các vị trí đứng trước `i` để ghép với `i` (đúng định nghĩa dãy yếu kết thúc tại `i`). Kết quả `16` khớp với `C(5,3)+C(5,4)+C(5,5)`.

In [24]:
def trace_duplicates(nums):
    n = len(nums)
    dp = [defaultdict(int) for _ in range(n)]
    ans = 0
    for i in range(n):
        reads, added = [], 0
        for j in range(i):
            d = nums[i] - nums[j]
            cnt = dp[j].get(d, 0)
            reads.append(cnt)
            ans += cnt
            added += cnt
            dp[i][d] += cnt + 1
        print(f"i={i}: dp[j][0] đọc được = {reads}, ans += {added} -> {ans}, dp[{i}] = {dict(dp[i])}")
    return ans

trace_duplicates([7, 7, 7, 7, 7])
n = 5
print("Kiểm tra công thức 2^n - 1 - n - C(n,2) =", 2**n - 1 - n - comb(n, 2))

i=0: dp[j][0] đọc được = [], ans += 0 -> 0, dp[0] = {}
i=1: dp[j][0] đọc được = [0], ans += 0 -> 0, dp[1] = {0: 1}
i=2: dp[j][0] đọc được = [0, 1], ans += 1 -> 1, dp[2] = {0: 3}
i=3: dp[j][0] đọc được = [0, 1, 3], ans += 4 -> 5, dp[3] = {0: 7}
i=4: dp[j][0] đọc được = [0, 1, 3, 7], ans += 11 -> 16, dp[4] = {0: 15}
Kiểm tra công thức 2^n - 1 - n - C(n,2) = 16


## 13. Code nộp LeetCode — `class Solution`

Trên LeetCode, `List` đã được import sẵn; trong notebook ta tự `from typing import List`. Bản nộp giữ đúng logic `arith_dp`, bỏ hết `print`.

In [25]:
from typing import List
from collections import defaultdict


class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        n = len(nums)
        dp = [defaultdict(int) for _ in range(n)]
        ans = 0
        for i in range(n):
            for j in range(i):
                d = nums[i] - nums[j]
                cnt = dp[j].get(d, 0)
                ans += cnt
                dp[i][d] += cnt + 1
        return ans


sol = Solution()

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

check("Ví dụ 1 [2,4,6,8,10]", sol.numberOfArithmeticSlices([2, 4, 6, 8, 10]), 7)
check("Ví dụ 2 [7,7,7,7,7]", sol.numberOfArithmeticSlices([7, 7, 7, 7, 7]), 16)
check("n = 1", sol.numberOfArithmeticSlices([5]), 0)
check("n = 2 (chỉ là cặp, không đếm)", sol.numberOfArithmeticSlices([1, 2]), 0)
check("không có dãy nào [1,1,2,5,7]", sol.numberOfArithmeticSlices([1, 1, 2, 5, 7]), 0)
check("công sai âm [1,2,3,2,1]", sol.numberOfArithmeticSlices([1, 2, 3, 2, 1]), 2)
check("giá trị trùng [2,2,3,4]", sol.numberOfArithmeticSlices([2, 2, 3, 4]), 2)
check("[3,3,3,3]", sol.numberOfArithmeticSlices([3, 3, 3, 3]), 5)
check("biên 32-bit [-2^31, 0, 2^31-1]", sol.numberOfArithmeticSlices([-2**31, 0, 2**31 - 1]), 0)
check("biên 32-bit [-2^31+1, 0, 2^31-1]", sol.numberOfArithmeticSlices([-2**31 + 1, 0, 2**31 - 1]), 1)

Ví dụ 1 [2,4,6,8,10]                   PASS (7)
Ví dụ 2 [7,7,7,7,7]                    PASS (16)
n = 1                                  PASS (0)
n = 2 (chỉ là cặp, không đếm)          PASS (0)
không có dãy nào [1,1,2,5,7]           PASS (0)
công sai âm [1,2,3,2,1]                PASS (2)
giá trị trùng [2,2,3,4]                PASS (2)
[3,3,3,3]                              PASS (5)
biên 32-bit [-2^31, 0, 2^31-1]         PASS (0)
biên 32-bit [-2^31+1, 0, 2^31-1]       PASS (1)


### 13.1. Test đối chiếu chéo 4 cách trên input ngẫu nhiên

Brute force đúng theo định nghĩa nên là "trọng tài". Ta sinh nhiều mảng nhỏ (độ dài 1..10) với giá trị trong khoảng hẹp `[-3, 3]` để **cố tình tạo nhiều giá trị trùng và nhiều cấp số cộng**, rồi so 4 cách + `Solution`. `random.seed` cố định để chạy lại luôn ra cùng kết quả.

In [26]:
random.seed(446)
n_tests = 300
for _ in range(n_tests):
    n = random.randint(1, 10)
    arr = [random.randint(-3, 3) for _ in range(n)]
    expected = arith_bruteforce(arr)
    results = {
        "dfs": arith_dfs(arr),
        "dfs_memo": arith_dfs_memo(arr),
        "dp": arith_dp(arr),
        "Solution": sol.numberOfArithmeticSlices(arr),
    }
    for name, got in results.items():
        assert got == expected, (name, arr, got, expected)
print(f"{n_tests} test ngẫu nhiên: cả 4 cách + Solution đều khớp brute force")

300 test ngẫu nhiên: cả 4 cách + Solution đều khớp brute force


### 13.2. Thử tốc độ với `n = 1000` (giới hạn đề)

Cách 4 có khoảng `n(n-1)/2 ≈ 500.000` cặp, mỗi cặp vài thao tác dict → chạy chỉ mất cỡ vài phần mười giây (dưới 1 giây) với Python. Ta thử hai kiểu dữ liệu: giá trị ngẫu nhiên rộng (ít cấp số cộng) và giá trị nằm trong khoảng nhỏ (rất nhiều cấp số cộng). Lưu ý kiểu thứ hai cho đáp án vượt 32-bit — LeetCode sẽ không cho input như vậy, nhưng Python vẫn tính đúng nhờ số nguyên không giới hạn.

In [27]:
random.seed(1)
big_random = [random.randint(-2**31, 2**31 - 1) for _ in range(1000)]
big_dense = [random.randint(0, 30) for _ in range(1000)]

for name, arr in [("ngẫu nhiên rộng", big_random), ("khoảng nhỏ [0,30]", big_dense)]:
    t0 = time.perf_counter()
    ans = sol.numberOfArithmeticSlices(arr)
    t1 = time.perf_counter()
    print(f"{name:<18} | n=1000 | đáp án có {len(str(ans))} chữ số | {t1 - t0:.2f}s")

ngẫu nhiên rộng    | n=1000 | đáp án có 1 chữ số | 0.18s


khoảng nhỏ [0,30]  | n=1000 | đáp án có 16 chữ số | 0.08s


## 14. Vì sao Cách 4 đúng?

**Invariant (bất biến) bằng lời:** sau khi xử lý xong vị trí `i`, với mọi công sai `d`, `dp[i][d]` đúng bằng số dãy yếu (dài ≥ 2) kết thúc tại `i` với công sai `d`; và `ans` đúng bằng số dãy thật (dài ≥ 3) có phần tử cuối ở vị trí ≤ `i`.

**Chứng minh ngắn:**

1. **Mỗi dãy yếu kết thúc tại `i` được đếm đúng một lần vào `dp[i][d]`.** Gọi `j` là vị trí kề trước `i` trong dãy. Bỏ `nums[i]`, phần còn lại là: hoặc chỉ một phần tử `nums[j]` (dãy là cặp `(j,i)` → đếm bởi `+1`), hoặc một dãy yếu kết thúc tại `j` cùng công sai `d` (đếm trong `dp[j][d]`). Mỗi dãy có **đúng một** `j` kề trước, nên rơi vào đúng một lượt `(j, i)`, không bị đếm hai lần.
2. **Mỗi dãy thật được cộng vào `ans` đúng một lần.** Một dãy thật có hai vị trí cuối duy nhất `(j, i)`. Bỏ phần tử cuối, ta được một dãy dài ≥ 2 = dãy yếu kết thúc tại `j` — được đếm trong `dp[j][d]` và cộng vào `ans` đúng ở lượt `(j, i)`.
3. **Không đếm nhầm dãy dài 2.** Dòng `ans += cnt` chỉ cộng số dãy yếu **được nối thêm** một phần tử, mà nối thêm thì độ dài ≥ 3. Phần `+1` (cặp) chỉ đi vào `dp`, không đi vào `ans`.
4. **Thứ tự hợp lệ:** `dp[j]` với `j < i` đã hoàn chỉnh trước khi được đọc (theo invariant ở bước `j`).

## 15. Độ phức tạp

Gọi `n = len(nums)`.

- **Time `O(n²)`**: hai vòng `for i`, `for j < i` → `n(n-1)/2` cặp; mỗi cặp một lần đọc `get` và một lần ghi dict, **trung bình** `O(1)` (bảng băm — hash table; trường hợp xấu lý thuyết của hash có thể chậm hơn, nhưng trên thực tế coi là `O(1)`).
- **Space `O(n²)`** trong trường hợp xấu: `dp[i]` có tối đa `i` key (mỗi `j < i` cho một công sai), tổng cộng tới ~`n²/2` key (với `n = 1000` là ~500.000 key — chấp nhận được).

Nhờ dùng `dp[j].get(d, 0)` thay cho `dp[j][d]` khi đọc, ta không chèn thêm key rỗng vào `dp[j]`, nên số key không vượt số cặp thật.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | `combinations` sinh mọi dãy con dài ≥ 3, kiểm tra | `O(2^n · n)` | `O(n)` | Dễ nhất, làm trọng tài test, chỉ chạy `n` ≤ ~18 |
| 2. DFS theo công sai | Chốt 2 phần tử đầu, chỉ nối phần tử đúng công sai | Xấu nhất hàm mũ (`[7]*n`) | `O(n)` stack | Cắt được nhánh hỏng, nhưng tính lặp trạng thái |
| 3. DFS + memo | Nhớ `extend(last, d)` | `O(n³)` | `O(n²)` | Hết hàm mũ, nhưng `10^9` với `n = 1000`, đệ quy sâu |
| 4. DP dãy yếu `dp[i][d]` | Nhìn về trước, mỗi cặp `(j,i)` xét 1 lần | `O(n²)` | `O(n²)` | **Lời giải chính**, pass LeetCode |

## 17. Các lỗi người mới hay gặp

### Lỗi 1: Đếm luôn cả dãy độ dài 2

```python
ans += cnt + 1          # SAI: cộng cả cặp (j, i) vào đáp án
```

Cặp chỉ là dãy yếu — nó phải vào `dp`, **không** vào `ans`. Kết quả sai lệch đúng bằng số cặp `n(n-1)/2`.

### Lỗi 2: Gán `=` thay vì cộng dồn `+=`

```python
dp[i][d] = cnt + 1      # SAI: j sau ghi đè j trước khi cùng công sai d
```

Chỉ lộ ra khi có giá trị trùng: với `[2,2,3,4]`, cả `j=0` và `j=1` (hai số 2) cho `d=1` tại số 3; ghi đè làm mất một cặp.

### Lỗi 3: `[defaultdict(int)] * n`

Mọi vị trí dùng chung **một** dict (mục 10.1) → số liệu của các `i` trộn lẫn.

### Lỗi 4: Cộng đáp án bằng `sum` của dp

```python
return sum(sum(row.values()) for row in dp)   # SAI: đây là tổng số DÃY YẾU
```

Bê nguyên thói quen `sum(dp)` từ #413 sang. Ở #446 `dp` đếm dãy yếu (có cả cặp), phải trừ đi các cặp mới ra đáp án.

Cell dưới chạy cả 4 bản sai để thấy rõ chúng hỏng ở đâu.

In [28]:
def wrong_count_pairs(nums):
    dp = [defaultdict(int) for _ in nums]; ans = 0
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]; cnt = dp[j].get(d, 0)
            ans += cnt + 1                    # Lỗi 1
            dp[i][d] += cnt + 1
    return ans

def wrong_assign(nums):
    dp = [defaultdict(int) for _ in nums]; ans = 0
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]; cnt = dp[j].get(d, 0)
            ans += cnt
            dp[i][d] = cnt + 1                # Lỗi 2
    return ans

def wrong_shared(nums):
    dp = [defaultdict(int)] * len(nums); ans = 0   # Lỗi 3
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]; cnt = dp[j].get(d, 0)
            ans += cnt
            dp[i][d] += cnt + 1
    return ans

def wrong_sum_dp(nums):
    dp = [defaultdict(int) for _ in nums]
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            dp[i][d] += dp[j].get(d, 0) + 1
    return sum(sum(row.values()) for row in dp)    # Lỗi 4

for arr in [[2, 4, 6, 8, 10], [2, 2, 3, 4], [7, 7, 7, 7, 7]]:
    print(f"{str(arr):<17} đúng={arith_dp(arr):>3} | lỗi1={wrong_count_pairs(arr):>3} "
          f"| lỗi2={wrong_assign(arr):>3} | lỗi3={wrong_shared(arr):>3} | lỗi4={wrong_sum_dp(arr):>3}")

[2, 4, 6, 8, 10]  đúng=  7 | lỗi1= 17 | lỗi2=  7 | lỗi3= 16 | lỗi4= 17
[2, 2, 3, 4]      đúng=  2 | lỗi1=  8 | lỗi2=  1 | lỗi3=  5 | lỗi4=  8
[7, 7, 7, 7, 7]   đúng= 16 | lỗi1= 26 | lỗi2= 10 | lỗi3=1013 | lỗi4= 26


### Lỗi 5 (C++/Java): tràn số khi tính công sai

Python có số nguyên **không giới hạn**, nên `nums[i] - nums[j]` luôn đúng. Nhưng ở C++/Java, `nums[i]` là `int` 32-bit trong `[-2^31, 2^31 - 1]`, còn hiệu có thể lên tới `(2^31 - 1) - (-2^31) = 2^32 - 1` → **tràn số (overflow)**: ở Java hiệu bị "quấn vòng" (wrap around) thành số sai, ở C++ tràn số có dấu còn là hành vi không xác định (undefined behavior).

Hậu quả: hai công sai khác nhau có thể bị quấn về **cùng** một giá trị và bị đếm nhầm.

Cách xử lý ở C++/Java:

```cpp
long long d = (long long)nums[i] - nums[j];      // tính bằng 64-bit
if (d > INT_MAX || d < INT_MIN) continue;        // công sai quá lớn: không thể có phần tử thứ 3
// dùng unordered_map<long long, int> (hoặc <int,int> sau khi đã lọc)
```

**Vì sao được phép bỏ qua cặp có `|d|` quá lớn?** Một dãy thật cần ít nhất 3 phần tử `a, a+d, a+2d`, tất cả nằm trong `[-2^31, 2^31-1]`. Khi đó `|2d| ≤ 2^32 - 1` → `d` phải nằm trong miền `int`. Cặp có `d` ngoài miền `int` không bao giờ nối dài được, và bản thân cặp thì không được đếm vào đáp án.

Ngoài ra số lượng dãy yếu trong `dp` có thể lớn hơn đáp án một chút (chứa cả cặp), nên nhiều người dùng `long` cho bộ đếm cho chắc. Cell dưới **mô phỏng** phép trừ 32-bit trong Python để thấy tràn số trông như thế nào.

In [29]:
def to_int32(x):
    # Mô phỏng cách số nguyên 32-bit có dấu "quấn vòng" (như Java)
    x &= 0xFFFFFFFF
    return x - 2**32 if x >= 2**31 else x

a, b = 2**31 - 1, -2**31
print("Python  :", a - b)            # 4294967295, đúng
print("int32   :", to_int32(a - b))  # -1, sai hoàn toàn!

# Hai công sai thật khác nhau nhưng quấn về cùng một giá trị 32-bit:
print("d thật = 1          ->", to_int32(1))
print("d thật = 1 - 2^32   ->", to_int32(1 - 2**32), "(giống hệt d = 1!)")

Python  : 4294967295
int32   : -1
d thật = 1          -> 1
d thật = 1 - 2^32   -> 1 (giống hệt d = 1!)


## 18. Mẹo nhận dạng pattern

- **"Đếm dãy con (subsequence) có tính chất phụ thuộc vào 2 phần tử liên tiếp"** (cấp số cộng, hiệu cố định, ...) → nghĩ ngay tới `dp[i][key]` = số dãy **kết thúc tại `i`** với "trạng thái" `key` (ở đây `key` = công sai). Mỗi bước nhìn về mọi `j < i`.
- **Trạng thái có miền giá trị rộng / âm** (công sai tới `±2^32`) → dùng **dict** thay cho mảng: `dp = [defaultdict(int) for _ in range(n)]`.
- **Đề yêu cầu độ dài tối thiểu `≥ k`** → đếm cả các dãy "chưa đủ dài" làm nguyên liệu (dãy yếu), và chỉ cộng vào đáp án ở **thời điểm dãy đạt đủ độ dài**. Đây là kỹ thuật "đếm có điều kiện độ dài" rất hay gặp.
- **Đếm theo vị trí, có giá trị trùng** → nhớ dùng `+=` để cộng dồn qua mọi `j`.
- Dấu hiệu của **#413** (mảng con liên tiếp) → chỉ cần 1 biến, `O(n)` thời gian `O(1)` bộ nhớ. Thấy chữ **subsequence** → chuyển sang khung `O(n²)` của bài này.

## 19. Biến thể và bài liên quan

| Bài | Khác gì | Ý tưởng |
|---|---|---|
| #413 Arithmetic Slices | mảng con liên tiếp | `cur = cur + 1` nếu cùng hiệu, ngược lại `0`; cộng dồn |
| #1027 Longest Arithmetic Subsequence | tìm **độ dài dài nhất** thay vì đếm | `dp[i][d] = dp[j].get(d, 1) + 1`, lấy `max` |
| #1218 Longest Arithmetic Subsequence of Given Difference | công sai **cho trước** | chỉ cần một dict theo **giá trị**: `best[x] = best.get(x - diff, 0) + 1`, `O(n)` |
| #115 Distinct Subsequences, #940 Distinct Subsequences II | đếm dãy con khác | cùng tinh thần "đếm dãy con bằng DP kết thúc tại vị trí" |

Cell dưới minh họa **#1027**: chỉ đổi phép "cộng số lượng" thành "lấy độ dài lớn nhất" — khung `dp[i][d]` giữ nguyên. (Ví dụ lấy từ đề #1027.)

In [30]:
def longest_arith_seq_length(nums):
    # #1027: dp[i][d] = độ dài dãy (>= 2) dài nhất kết thúc tại i với công sai d
    dp = [dict() for _ in nums]
    best = 2 if len(nums) >= 2 else len(nums)
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            dp[i][d] = max(dp[i].get(d, 0), dp[j].get(d, 1) + 1)   # dp[j] mặc định 1: chỉ có nums[j]
            best = max(best, dp[i][d])
    return best

print(longest_arith_seq_length([3, 6, 9, 12]))            # 4
print(longest_arith_seq_length([9, 4, 7, 2, 10]))         # 3
print(longest_arith_seq_length([20, 1, 15, 3, 10, 5, 8])) # 4

4
3
4


## 20. Bài tập vận dụng

Làm theo thứ tự từ dễ đến khó. Các cell TODO chỉ **định nghĩa hàm**, test được để ở dạng comment — làm xong hãy bỏ comment để tự kiểm tra. Gợi ý ở mục 21, đáp án ở **cuối notebook**.

### Bài 1 — Đoán output (không chạy code)

Hàm `numberOfArithmeticSlices` trả về bao nhiêu với mỗi input sau? Ghi kèm lý do.

```python
a) [1, 2, 3, 4]
b) [3, 3, 3, 3]
c) [1, 2]
d) [1, 3, 5, 4, 7]
```

### Bài 2 — Tính tay bảng DP

Lập bảng trace giống mục 12 cho `nums = [2, 2, 3, 4]`: với từng cặp `(j, i)` ghi `d`, `dp[j][d]` trước, `ans` sau, `dp[i][d]` sau. Cho biết `dp[3]` cuối cùng và đáp án. Giải thích vì sao đáp án không phải 1.

### Bài 3 — Điền TODO cho lời giải chính

Hoàn thiện hàm `exercise_arith_dp` theo đúng khung dãy yếu.

In [31]:
def exercise_arith_dp(nums):
    n = len(nums)
    # TODO 1: tạo dp là list gồm n defaultdict(int) ĐỘC LẬP
    # TODO 2: ans = 0
    # TODO 3: duyệt i, rồi j < i, tính d = nums[i] - nums[j]
    # TODO 4: cnt = số dãy yếu kết thúc tại j với công sai d (đọc bằng .get)
    # TODO 5: cộng cnt vào ans; cộng cnt + 1 vào dp[i][d]
    raise NotImplementedError

# Làm xong hãy bỏ comment:
# assert exercise_arith_dp([2, 4, 6, 8, 10]) == 7
# assert exercise_arith_dp([7, 7, 7, 7, 7]) == 16
# assert exercise_arith_dp([2, 2, 3, 4]) == 2

### Bài 4 — Tự viết lại #413 với `O(1)` bộ nhớ

Viết `arithmetic_slices_413(nums)` đếm số **mảng con liên tiếp** dài ≥ 3 là cấp số cộng, chỉ dùng một biến `cur` (số đoạn tốt kết thúc tại vị trí hiện tại) thay cho cả mảng `dp`.

Ví dụ: `[1,2,3,4] -> 3`, `[1] -> 0`, `[1,3,5,7,9] -> 6`, `[1,2,4,6,8] -> 3`.

In [32]:
def arithmetic_slices_413(nums):
    # TODO: cur = 0, total = 0; duyệt i từ 2; nếu 3 số cuối cùng hiệu thì cur += 1, ngược lại cur = 0
    pass

# assert arithmetic_slices_413([1, 2, 3, 4]) == 3
# assert arithmetic_slices_413([1]) == 0
# assert arithmetic_slices_413([1, 3, 5, 7, 9]) == 6
# assert arithmetic_slices_413([1, 2, 4, 6, 8]) == 3

### Bài 5 — Biến thể: công sai cho trước, đạt `O(n)`

Viết `count_slices_with_diff(nums, diff)` đếm số dãy con cấp số cộng (dài ≥ 3) **có công sai đúng bằng `diff`**.

Yêu cầu: chạy `O(n)` bằng cách lưu theo **giá trị** thay vì theo vị trí (vì công sai đã cố định, phần tử kề trước của `x` bắt buộc có giá trị `x - diff`).

Ví dụ:

```python
count_slices_with_diff([2, 4, 6, 8, 10], 2) -> 6     # 7 dãy trừ [2,6,10]
count_slices_with_diff([2, 4, 6, 8, 10], 4) -> 1     # [2,6,10]
count_slices_with_diff([7, 7, 7, 7, 7], 0)  -> 16
count_slices_with_diff([2, 2, 3, 4], 1)     -> 2
```

In [33]:
def count_slices_with_diff(nums, diff):
    # TODO: single[v] = số lần giá trị v đã xuất hiện (dãy dài 1 kết thúc bằng v)
    # TODO: weak[v]   = số dãy yếu (dài >= 2) kết thúc bằng giá trị v, công sai diff
    # TODO: với mỗi x: đọc weak[x - diff], single[x - diff] TRƯỚC khi cập nhật (cẩn thận diff = 0!)
    pass

# assert count_slices_with_diff([2, 4, 6, 8, 10], 2) == 6
# assert count_slices_with_diff([2, 4, 6, 8, 10], 4) == 1
# assert count_slices_with_diff([7, 7, 7, 7, 7], 0) == 16
# assert count_slices_with_diff([2, 2, 3, 4], 1) == 2

### Bài 6 — Mini challenge: thống kê theo từng công sai

Viết `slices_by_diff(nums)` trả về **dict `{công sai: số dãy con cấp số cộng dài ≥ 3 có công sai đó}`** (bỏ các công sai có 0 dãy). Tổng các giá trị phải bằng `numberOfArithmeticSlices(nums)`.

Yêu cầu: `O(n²)`, và tự viết thêm test đối chiếu với brute force trên 200 mảng ngẫu nhiên nhỏ.

Ví dụ:

```python
slices_by_diff([2, 4, 6, 8, 10]) -> {2: 6, 4: 1}
slices_by_diff([1, 2, 3, 2, 1])  -> {1: 1, -1: 1}
slices_by_diff([7, 7, 7, 7, 7])  -> {0: 16}
```

In [34]:
def slices_by_diff(nums):
    # TODO: giống arith_dp, nhưng thay vì một biến ans, dùng một dict per_d
    #       cộng cnt vào per_d[d] tại đúng chỗ arith_dp làm ans += cnt
    pass

# assert slices_by_diff([2, 4, 6, 8, 10]) == {2: 6, 4: 1}
# assert slices_by_diff([1, 2, 3, 2, 1]) == {1: 1, -1: 1}
# assert slices_by_diff([7, 7, 7, 7, 7]) == {0: 16}

## 21. Gợi ý theo 3 mức

**Bài 1**
- Mức 1: Liệt kê theo độ dài 3, 4, 5... và nhớ đếm theo vị trí.
- Mức 2: b) là mảng toàn số giống nhau → dùng `C(4,3) + C(4,4)`.
- Mức 3: d) thử các bộ ba có công sai 2 và 3: `[1,3,5]`, `[3,5,7]`, `[1,4,7]`, `[1,3,5,7]`? (để ý `7` nằm sau `5`, còn `4` nằm giữa).

**Bài 2**
- Mức 1: có 6 cặp `(j, i)`; làm lần lượt `i = 1, 2, 3`.
- Mức 2: tại `i = 2` (số 3), cả `j = 0` và `j = 1` đều cho `d = 1` → `dp[2][1]` được cộng **hai lần**.
- Mức 3: tại `i = 3` (số 4), cặp `j = 2` đọc `dp[2][1] = 2` → `ans` tăng 2.

**Bài 3**
- Mức 1: chép lại khung ở mục 11 bằng trí nhớ.
- Mức 2: list comprehension `[defaultdict(int) for _ in range(n)]`.
- Mức 3: `cnt = dp[j].get(d, 0)`; `ans += cnt`; `dp[i][d] += cnt + 1`; cuối cùng `return ans`.

**Bài 4**
- Mức 1: `cur` đóng vai `dp[i-1]` ở mục 2.1.
- Mức 2: `for i in range(2, len(nums))`.
- Mức 3: `if nums[i]-nums[i-1] == nums[i-1]-nums[i-2]: cur += 1 else: cur = 0`, rồi `total += cur`.

**Bài 5**
- Mức 1: với công sai cố định, phần tử đứng trước `x` trong dãy phải có giá trị `p = x - diff`.
- Mức 2: số dãy yếu mới kết thúc tại `x` = `weak[p] + single[p]` (nối vào dãy yếu, hoặc tạo cặp với một lần xuất hiện của `p`); số dãy thật mới = `weak[p]`.
- Mức 3: `w, s = weak.get(p, 0), single.get(p, 0)` **trước**, rồi `ans += w`, `weak[x] = weak.get(x, 0) + w + s`, `single[x] = single.get(x, 0) + 1`. Với `diff = 0` thì `p == x` nên bắt buộc phải đọc trước khi ghi.

**Bài 6**
- Mức 1: đáp án tổng là tổng của mọi lần `ans += cnt`; mỗi lần đó gắn với một `d`.
- Mức 2: dùng `per_d = defaultdict(int)`; `per_d[d] += cnt`.
- Mức 3: trả `{d: c for d, c in per_d.items() if c > 0}`; test bằng cách lọc kết quả brute force theo công sai `seq[1]-seq[0]`.

## 22. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- a) `[1,2,3,4]` → **3**: `[1,2,3]`, `[2,3,4]`, `[1,2,3,4]`.
- b) `[3,3,3,3]` → **5**: `C(4,3) + C(4,4) = 4 + 1`.
- c) `[1,2]` → **0**: chưa đủ 3 phần tử.
- d) `[1,3,5,4,7]` → **4**: `[1,3,5]` (d=2), `[3,5,7]` (d=2), `[1,3,5,7]` (d=2), `[1,4,7]` (d=3).

### Đáp án Bài 2 — `nums = [2, 2, 3, 4]`

| i (val) | j (val) | d | dp[j][d] trước | ans sau | dp[i][d] sau |
|---|---|---:|---:|---:|---:|
| 1 (2) | 0 (2) | 0 | 0 | 0 | 1 |
| 2 (3) | 0 (2) | 1 | 0 | 0 | 1 |
| 2 (3) | 1 (2) | 1 | 0 | 0 | 2 |
| 3 (4) | 0 (2) | 2 | 0 | 0 | 1 |
| 3 (4) | 1 (2) | 2 | 0 | 0 | 2 |
| 3 (4) | 2 (3) | 1 | 2 | **2** | 3 |

`dp[3] = {2: 2, 1: 3}`, đáp án **2**: `[2,3,4]` với số 2 ở vị trí 0, và `[2,3,4]` với số 2 ở vị trí 1. Không phải 1 vì đề đếm theo **vị trí** — đây cũng là lý do bắt buộc dùng `+=` ở `dp[2][1]`. Cell dưới kiểm chứng bằng `trace_arith_dp`.

In [35]:
answer_bai1 = {"a": 3, "b": 5, "c": 0, "d": 4}
for key, arr in zip("abcd", [[1, 2, 3, 4], [3, 3, 3, 3], [1, 2], [1, 3, 5, 4, 7]]):
    assert arith_bruteforce(arr) == answer_bai1[key] == arith_dp(arr)
print("Bài 1 đúng:", answer_bai1)

dp_b2, ans_b2 = trace_arith_dp([2, 2, 3, 4])
assert ans_b2 == 2 and dict(dp_b2[3]) == {2: 2, 1: 3}
print("Bài 2 đúng: dp[3] =", dict(dp_b2[3]))

Bài 1 đúng: {'a': 3, 'b': 5, 'c': 0, 'd': 4}
 i(val)  j(val)    d  dp[j][d]   ans  dp[i][d]
  1( 2)   0( 2)    0         0     0         1
  2( 3)   0( 2)    1         0     0         1
  2( 3)   1( 2)    1         0     0         2
  3( 4)   0( 2)    2         0     0         1
  3( 4)   1( 2)    2         0     0         2
  3( 4)   2( 3)    1         2     2         3
Đáp án: 2
Bài 2 đúng: dp[3] = {2: 2, 1: 3}


In [36]:
# ===== Đáp án Bài 3 và Bài 4 =====
def answer_exercise_arith_dp(nums):
    n = len(nums)
    dp = [defaultdict(int) for _ in range(n)]
    ans = 0
    for i in range(n):
        for j in range(i):
            d = nums[i] - nums[j]
            cnt = dp[j].get(d, 0)
            ans += cnt
            dp[i][d] += cnt + 1
    return ans

assert answer_exercise_arith_dp([2, 4, 6, 8, 10]) == 7
assert answer_exercise_arith_dp([7, 7, 7, 7, 7]) == 16
assert answer_exercise_arith_dp([2, 2, 3, 4]) == 2


def answer_arithmetic_slices_413(nums):
    cur = total = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1          # kéo dài mọi đoạn cũ + 1 đoạn mới dài 3
        else:
            cur = 0
        total += cur
    return total

assert answer_arithmetic_slices_413([1, 2, 3, 4]) == 3
assert answer_arithmetic_slices_413([1]) == 0
assert answer_arithmetic_slices_413([1, 3, 5, 7, 9]) == 6
assert answer_arithmetic_slices_413([1, 2, 4, 6, 8]) == 3

random.seed(413)
for _ in range(200):
    arr = [random.randint(-2, 2) for _ in range(random.randint(1, 12))]
    assert answer_arithmetic_slices_413(arr) == arithmetic_slices_413_naive(arr)
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


### Đáp án Bài 5 và Bài 6

Bài 5 đạt `O(n)` vì khi công sai cố định, "phần tử đứng trước" của `x` chỉ có thể mang **một** giá trị `x - diff`; mọi lần xuất hiện của giá trị đó có thể gộp chung một bộ đếm. Bài 6 giữ nguyên khung `O(n²)`, chỉ tách `ans` theo công sai.

In [37]:
def answer_count_slices_with_diff(nums, diff):
    single = {}   # single[v]: số lần giá trị v đã xuất hiện
    weak = {}     # weak[v]  : số dãy yếu (dài >= 2, công sai diff) kết thúc bằng giá trị v
    ans = 0
    for x in nums:
        p = x - diff
        w, s = weak.get(p, 0), single.get(p, 0)     # ĐỌC trước (quan trọng khi diff = 0)
        ans += w
        weak[x] = weak.get(x, 0) + w + s
        single[x] = single.get(x, 0) + 1
    return ans

assert answer_count_slices_with_diff([2, 4, 6, 8, 10], 2) == 6
assert answer_count_slices_with_diff([2, 4, 6, 8, 10], 4) == 1
assert answer_count_slices_with_diff([7, 7, 7, 7, 7], 0) == 16
assert answer_count_slices_with_diff([2, 2, 3, 4], 1) == 2


def answer_slices_by_diff(nums):
    n = len(nums)
    dp = [defaultdict(int) for _ in range(n)]
    per_d = defaultdict(int)
    for i in range(n):
        for j in range(i):
            d = nums[i] - nums[j]
            cnt = dp[j].get(d, 0)
            per_d[d] += cnt
            dp[i][d] += cnt + 1
    return {d: c for d, c in per_d.items() if c > 0}

assert answer_slices_by_diff([2, 4, 6, 8, 10]) == {2: 6, 4: 1}
assert answer_slices_by_diff([1, 2, 3, 2, 1]) == {1: 1, -1: 1}
assert answer_slices_by_diff([7, 7, 7, 7, 7]) == {0: 16}


def brute_by_diff(nums):
    n, res = len(nums), defaultdict(int)
    for length in range(3, n + 1):
        for idx in combinations(range(n), length):
            seq = [nums[t] for t in idx]
            if is_arithmetic(seq):
                res[seq[1] - seq[0]] += 1
    return dict(res)

random.seed(2024)
for _ in range(200):
    arr = [random.randint(-3, 3) for _ in range(random.randint(1, 9))]
    by = answer_slices_by_diff(arr)
    assert by == brute_by_diff(arr)
    assert sum(by.values()) == arith_dp(arr)
    for d in range(-6, 7):
        assert answer_count_slices_with_diff(arr, d) == by.get(d, 0)
print("Bài 5, Bài 6: PASS (đối chiếu brute force 200 mảng ngẫu nhiên)")

Bài 5, Bài 6: PASS (đối chiếu brute force 200 mảng ngẫu nhiên)


## 23. Template ngắn dễ nhớ

```python
from collections import defaultdict

class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        dp = [defaultdict(int) for _ in nums]   # dp[i][d] = số DÃY YẾU (dài >= 2) kết thúc tại i
        ans = 0
        for i in range(len(nums)):
            for j in range(i):
                d = nums[i] - nums[j]
                cnt = dp[j].get(d, 0)
                ans += cnt                      # nối thêm nums[i] -> dãy dài >= 3
                dp[i][d] += cnt + 1             # + cặp (j, i)
        return ans
```

Câu thần chú: **"dp đếm dãy yếu, ans chỉ đếm phần được nối dài."**

## 24. Checklist trước khi submit

- Mình có hiểu `dp[i][d]` đếm **dãy yếu (dài ≥ 2)**, còn `ans` chỉ cộng `dp[j][d]` (không cộng `+1`) chưa?
- `dp` có được tạo bằng **list comprehension** (n dict độc lập), không phải `[defaultdict(int)] * n`?
- Có dùng `+=` (cộng dồn qua mọi `j`) cho `dp[i][d]` để đúng khi có **giá trị trùng** như `[7,7,7,7,7]`, `[2,2,3,4]`?
- Đọc `dp[j]` bằng `.get(d, 0)` để không chèn key rỗng?
- Vòng trong là `for j in range(i)` (chỉ nhìn về **trước**)?
- Đã thử: Ví dụ 1 (`7`), Ví dụ 2 (`16`), `n = 1`, `n = 2`, công sai âm, giá trị biên `±2^31`?
- Nếu viết C++/Java: công sai tính bằng `long long`/`long`, bỏ qua cặp có công sai ngoài miền `int`?
- Đã đối chiếu với brute force `combinations` trên mảng nhỏ ngẫu nhiên chưa?